In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('california_restaurants_cleaned.csv')
df.head()

,Yelp URL,Name,Street Address,Zip Code,City,State,Price Range,Phone,Rating,Number of Reviews,Website,Menu Link,Image 1,Image 2,Image 3,Category 1,Category 2,Category 3,review_outliers
0,https://www.yelp.com/biz/in-s%C4%ABt-coffee-bu...,In-sīt Coffee,6930 Beach Blvd Ste L301,90621,Buena Park,CA,$,(714) 670 6958,4.0,583.0,NaN,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/TgX8pH...,https://s3-media0.fl.yelpcdn.com/bphoto/e9oSgM...,https://s3-media0.fl.yelpcdn.com/bphoto/gJUxfa...,Coffee & Tea,Waffles,Breakfast & Brunch,False
1,https://www.yelp.com/biz/julies-cafe-chino-3,Julie's Cafe,3746 Riverside Dr,91710,Chino,CA,$$,(909) 992 0013,4.5,179.0,www.juliescafes.com,www.juliescafes.com/menu/,https://s3-media0.fl.yelpcdn.com/bphoto/__RSTK...,https://s3-media0.fl.yelpcdn.com/bphoto/bdPEPF...,https://s3-media0.fl.yelpcdn.com/bphoto/7d4bRB...,Bagels,Breakfast & Brunch,Cafes,False
2,https://www.yelp.com/biz/bonanza-bakery-and-ca...,Bonanza Bakery & Cafe,"20657 Golden Springs Dr Ste - 104,105",91789,Diamond Bar,CA,$$,(909) 274 7121,4.0,76.0,www.bonanzabakerycafe.com,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/ca8BLo...,https://s3-media0.fl.yelpcdn.com/bphoto/FVqd4a...,https://s3-media0.fl.yelpcdn.com/bphoto/3iL4th...,Desserts,Coffee & Tea,Breakfast & Brunch,False
3,https://www.yelp.com/biz/s-and-j-gran-cafe-dia...,S & J Gran Cafe,21050 Golden Springs Dr Ste C108,91789,Diamond Bar,CA,$$,(909) 551 0021,4.0,234.0,NaN,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/a_zwij...,https://s3-media0.fl.yelpcdn.com/bphoto/lbO1Z_...,https://s3-media0.fl.yelpcdn.com/bphoto/wPAAU4...,Cafes,Breakfast & Brunch,NaN,False
4,https://www.yelp.com/biz/sul-and-beans-rowland...,Sul & Beans,1330 Fullerton Rd Unit 110,91748,Rowland Heights,CA,$$,(626) 581 1581,4.0,643.0,www.sulandbeans.com,www.sulandbeans.com/menu,https://s3-media0.fl.yelpcdn.com/bphoto/l4NVad...,https://s3-media0.fl.yelpcdn.com/bphoto/TC-TF4...,https://s3-media0.fl.yelpcdn.com/bphoto/ku9I_Q...,Desserts,Shaved Ice,Coffee & Tea,False


In [2]:
df['high_rating'] = (df['Rating'] > 4).astype(int)
df['high_rating'].value_counts()

,count
high_rating,
0,334
1,325


In [8]:
features = df[['Price Range', 'Number of Reviews', 'Category 1', 'City']].copy()

features['Price Range'] = features['Price Range'].fillna('Unknown')
features['Category 1'] = features['Category 1'].fillna('Unknown')
features['City'] = features['City'].fillna('Unknown')
features['Number of Reviews'] = features['Number of Reviews'].fillna(0)

le_price = LabelEncoder()
le_category = LabelEncoder()
le_city = LabelEncoder()

features['Price Range'] = le_price.fit_transform(features['Price Range'].astype(str))
features['Category 1'] = le_category.fit_transform(features['Category 1'].astype(str))
features['City'] = le_city.fit_transform(features['City'].astype(str))

features.head()

,Price Range,Number of Reviews,Category 1,City
0,0,583.0,25,3
1,1,179.0,4,5
2,1,76.0,32,13
3,1,234.0,18,13
4,1,643.0,32,29


In [4]:
X = features
y = df['high_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training size: {len(X_train)}")
print(f"Testing size: {len(X_test)}")

Training size: 527
Testing size: 132


In [5]:
baseline_pred = [1] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"Baseline Accuracy (predict all high rating): {baseline_accuracy:.2f}")

Baseline Accuracy (predict all high rating): 0.56


In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Improved Model Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Improved Model Accuracy: 0.49

Classification Report:
              precision    recall  f1-score   support

           0       0.45      0.69      0.54        58
           1       0.58      0.34      0.43        74

    accuracy                           0.49       132
   macro avg       0.52      0.51      0.49       132
weighted avg       0.52      0.49      0.48       132



In [10]:
improved_accuracy = accuracy_score(y_test, y_pred)
print("=== Model Comparison ===")
print(f"Baseline Accuracy:        {baseline_accuracy:.2f}")
print(f"Logistic Regression:      {improved_accuracy:.2f}")
if improved_accuracy > baseline_accuracy:
    print("\nThe logistic regression model outperforms the naive baseline.")
else:
    print("\nThe logistic regression did not outperform the baseline, suggesting these features alone are not strong predictors of rating.")

=== Model Comparison ===
Baseline Accuracy:        0.56
Logistic Regression:      0.49

The logistic regression did not outperform the baseline, suggesting these features alone are not strong predictors of rating.
